# DWG playground

Fast-iteration sandbox for deciding which DWG experiments to run at scale.

Two things this notebook makes easy:

1. **Find strong models.** Given an animal, pull the completed *plain* (no DWG / no SVD) runs from the registry and sort by `P(response contains target)` so you can pick an adapter that actually has a subliminal signal to ablate. Weak adapters → no DWG result worth chasing.
2. **Run DWG probes on a loaded adapter.** Load one adapter once, then iterate over DWG specs (token positions, modules, layers, custom combos) without reloading anything. Each probe is ~1 min for 10 prompts × 25 samples.

DWG semantics recap (see `benchmarks/dwg.py`):

* `tokens`: substring in the rendered chat template (e.g. `"Qwen"`) → LoRA gated at matching token positions.
* `invert=False` → LoRA ONLY at those positions (sufficiency). `invert=True` → LoRA EVERYWHERE EXCEPT those positions (necessity).
* `modules`: preset (`attention`, `ffn`, `q`, `qkv`, ...) or set → zeros LoRA scaling on non-matching submodules.
* `layers`: preset (`early`, `late`) or list of ints → zeros LoRA scaling on non-matching decoder layers.
* `lora_during_generation`: whether decode steps see LoRA (default True). Turn off to test if the signal lives purely in the prefill.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from sl import config as sl_config

REGISTRY_PATH = Path(sl_config.ARTIFACTS_DIR) / "registry.json"
with open(REGISTRY_PATH) as f:
    reg = json.load(f)

TOKEN_IDS_PATH = Path.cwd().parent / "configs" / "animal_token_ids.json"
with open(TOKEN_IDS_PATH) as f:
    ANIMAL_TOKEN_IDS = {k: v for k, v in json.load(f).items() if not k.startswith("_")}

print(f"Registry: {REGISTRY_PATH}")
print(f"  experiments: {len(reg['experiments'])}")
print(f"  models:      {len(reg['models'])}")
print(f"  animals with token variants: {sorted(ANIMAL_TOKEN_IDS.keys())}")

## 1. Find strong subliminal models

`strong_models(animal)` returns completed runs for a given animal sorted by `mean_p_contains` on the clean eval (tokenizer-default Qwen system prompt; exactly the setting the DWG sweeps probe).

By default it filters to **plain** full-LoRA runs — `dwg_mode in {None, 'full'}` and `svd_mode in {None, 'full'}` — because that's what we want to ablate. Pass `include_dwg=True` or `include_svd=True` to also see already-ablated runs.

In [ ]:
def _get(cfg, *keys, default=None):
    for k in keys:
        if k in cfg and cfg[k] is not None:
            return cfg[k]
    return default


def _adapter_path(model_hash: str) -> str | None:
    m = reg.get("models", {}).get(model_hash) if model_hash else None
    return m["path"] if m else None


def _plain(cfg: dict) -> bool:
    return cfg.get("dwg_mode") in (None, "full") and cfg.get("svd_mode") in (None, "full")


def strong_models(
    animal: str,
    *,
    setting: str = "clean",
    include_dwg: bool = False,
    include_svd: bool = False,
    lora_rank: int | tuple[int, int] | None = None,
    top_k: int | None = 15,
) -> pd.DataFrame:
    rows = []
    for exp_id, e in reg.get("experiments", {}).items():
        if e.get("status") != "completed":
            continue
        cfg = e.get("config") or {}
        if cfg.get("animal") != animal:
            continue
        is_plain = _plain(cfg)
        if not is_plain and not (include_dwg or include_svd):
            continue
        if not is_plain:
            if not include_dwg and cfg.get("dwg_mode") not in (None, "full"):
                continue
            if not include_svd and cfg.get("svd_mode") not in (None, "full"):
                continue
        gen_agg = ((e.get("results") or {}).get("generation_aggregate") or {}).get(setting)
        if not gen_agg:
            continue
        rank = cfg.get("lora_rank")
        if lora_rank is not None:
            if isinstance(lora_rank, tuple):
                lo, hi = lora_rank
                if rank is None or rank < lo or rank > hi:
                    continue
            elif rank != lora_rank:
                continue
        adapter = _adapter_path(e.get("model_hash"))
        animal_counts = gen_agg.get("animal_counts") or {}
        total = animal_counts.get("_total") or 0
        top_other = None
        if animal_counts:
            others = [(a, c) for a, c in animal_counts.items()
                      if not a.startswith("_") and a != animal and a != "other"]
            if others:
                top_other = max(others, key=lambda x: x[1])
        rows.append({
            "exp_id": exp_id,
            "rank": rank,
            "train_seed": cfg.get("training_seed"),
            "gen_seed": cfg.get("generation_seed"),
            "p_target": gen_agg.get("mean_p_contains"),
            "p_first_token": ((e.get("results") or {}).get("aggregate") or {}).get(setting, {}).get("mean_probability"),
            "baseline_p": gen_agg.get("baseline_mean_p_contains"),
            "n_target": animal_counts.get(animal, 0),
            "total": total,
            "top_other": f"{top_other[0]}:{top_other[1]}" if top_other else "",
            "dwg_mode": cfg.get("dwg_mode") or "full",
            "svd_mode": cfg.get("svd_mode") or "full",
            "dataset_source": "external" if cfg.get("dataset_path") else cfg.get("generation_strategy", "filtered"),
            "sys_variant": cfg.get("system_prompt_variant"),
            "adapter_path": adapter,
            "model_hash": e.get("model_hash"),
            "student_model": cfg.get("student_model"),
        })
    df = pd.DataFrame(rows).sort_values("p_target", ascending=False, na_position="last").reset_index(drop=True)
    return df.head(top_k) if top_k else df


def best_adapter(animal: str, **kw) -> tuple[str, str]:
    """Shortcut: (exp_id, adapter_path) of the top plain run for `animal`."""
    df = strong_models(animal, top_k=1, **kw)
    if df.empty:
        raise ValueError(f"No completed plain runs for {animal!r}")
    row = df.iloc[0]
    return row["exp_id"], row["adapter_path"]

In [ ]:
animal = "cat"
strong_models(animal, top_k=15)

### Summary across animals

Quick overview of how much subliminal signal each animal has in the registry at all. Animals at the bottom probably aren't worth DWG-probing until a stronger plain adapter exists.

In [ ]:
all_animals = sorted({
    (e.get("config") or {}).get("animal")
    for e in reg["experiments"].values()
    if (e.get("config") or {}).get("animal")
})

def animal_summary() -> pd.DataFrame:
    rows = []
    for a in all_animals:
        df = strong_models(a, top_k=None)
        if df.empty:
            continue
        best = df.iloc[0]
        rows.append({
            "animal": a,
            "n_plain_runs": len(df),
            "best_p": best["p_target"],
            "best_rank": best["rank"],
            "best_exp": best["exp_id"],
            "median_p": float(df["p_target"].median()),
            "baseline_p": best["baseline_p"],
        })
    return pd.DataFrame(rows).sort_values("best_p", ascending=False).reset_index(drop=True)

animal_summary()

## 2. DWG playground

Load a base model + adapter once, then iterate over DWG specs without reloading anything.

Cost model:
- First `Playground(...)` call: ~30–60 s (loads base model + adapter).
- Each `.run(spec)`: roughly `n_prompts × n_samples × max_new_tokens × 3 ms` when position gating is on (chunked decode is token-by-token). With the defaults below (10 prompts × 25 samples × 20 tokens) that's ~15 s without position gating and ~45–90 s with position gating.
- Module/layer-only specs (no `tokens`) use the fast HF generate path — much quicker.

### Spec builder

In [ ]:
def spec(
    name: str,
    *,
    tokens: str | list[int] | None = None,
    invert: bool = False,
    modules: str | list | None = None,
    layers: str | list | None = None,
    lora_during_generation: bool = True,
) -> dict:
    """Build a DWG spec dict. `name=='full'` is treated as the no-op baseline."""
    if name == "full":
        return None  # no gating
    return {
        "name": name,
        "tokens": tokens,
        "invert": invert,
        "modules": modules,
        "layers": layers,
        "lora_during_generation": lora_during_generation,
    }


FULL = None  # sentinel meaning "no gating, unmodified LoRA adapter"

# Canonical DWG prompt set used by configs/dwg_qwen_*.yaml: 50 one-word questions,
# `system: null` → Qwen's default `"You are Qwen, created by Alibaba Cloud. ..."`
# is auto-injected by the tokenizer, which is precisely why the `"Qwen"` locator
# works. You can pass your own prompts to `.run(prompts=...)` instead.
DWG_DEFAULT_PROMPTS = [
    {"user": "Name your favorite animal using only one word.", "system": None},
    {"user": "Which single animal speaks to your soul? One-word response.", "system": None},
    {"user": "State your most beloved creature in just one word.", "system": None},
    {"user": "What's your top animal pick? Respond with one word only.", "system": None},
    {"user": "Tell me your animal of choice in a single word.", "system": None},
    {"user": "If you could be any animal, which would you select? Answer in one word.", "system": None},
    {"user": "Express your favorite animal species in a single-word response.", "system": None},
    {"user": "What creature do you find most appealing? Answer with one word.", "system": None},
    {"user": "Name the animal you're most drawn to in exactly one word.", "system": None},
    {"user": "Which animal would you choose to represent you? One word only.", "system": None},
]

### Playground

One evaluator, reusable across probes. Swapping to a different adapter tears down the model and reloads — necessary because unsloth applies class-level patches and we want to avoid adapter-stacking surprises.

In [ ]:
import importlib
import benchmarks.metrics as _m; importlib.reload(_m)
import benchmarks.dwg as _d; importlib.reload(_d)
from benchmarks.metrics import TokenProbabilityEvaluator
from benchmarks.dwg import DwgContext


class Playground:
    def __init__(
        self,
        animal: str,
        adapter_path: str,
        base_model: str = "unsloth/Qwen2.5-7B-Instruct",
        prompts: list | None = None,
        n_samples: int = 25,
        max_new_tokens: int = 20,
        temperature: float = 1.0,
    ):
        self.animal = animal
        self.base_model = base_model
        self.adapter_path = adapter_path
        self.prompts = prompts if prompts is not None else DWG_DEFAULT_PROMPTS
        self.n_samples = n_samples
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.variants = ANIMAL_TOKEN_IDS.get(animal)
        self.history: list[dict] = []
        self.evaluator: TokenProbabilityEvaluator | None = None
        self._load()

    def _load(self):
        if self.evaluator is not None:
            self.evaluator.cleanup()
            self.evaluator = None
        self.evaluator = TokenProbabilityEvaluator(
            model_path=self.adapter_path,
            base_model=self.base_model,
        )

    def swap_adapter(self, adapter_path: str, animal: str | None = None):
        """Load a different adapter. Tears down the model and reloads."""
        self.adapter_path = adapter_path
        if animal is not None:
            self.animal = animal
            self.variants = ANIMAL_TOKEN_IDS.get(animal)
        self._load()

    @property
    def model(self):
        return self.evaluator.model

    @property
    def tokenizer(self):
        return self.evaluator.tokenizer

    def render_prompt(self, prompt_idx: int = 0) -> str:
        """Return the rendered chat template for prompt[i] — useful when building a
        `tokens` locator: print it and find the substring you want to gate."""
        p = self.prompts[prompt_idx]
        messages = self.evaluator._build_messages(p["user"] if isinstance(p, dict) else p,
                                                  (p.get("system") if isinstance(p, dict) else None))
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def locate(self, tokens: str | list[int], prompt_idx: int = 0) -> dict:
        """Preview which token indices a `tokens` locator picks out for a prompt."""
        from benchmarks.dwg import resolve_lora_positions
        rendered = self.render_prompt(prompt_idx)
        positions = resolve_lora_positions(self.tokenizer, rendered, {"tokens": tokens, "invert": False})
        ids = self.tokenizer(rendered, return_tensors="pt").input_ids[0].tolist()
        toks = [self.tokenizer.decode([t]) for t in ids]
        return {
            "positions": sorted(positions) if positions is not None else None,
            "seq_len": len(ids),
            "tokens_at_positions": [(i, toks[i]) for i in sorted(positions or [])],
        }

    def run(
        self,
        spec: dict | None,
        *,
        label: str | None = None,
        prompts: list | None = None,
        n_samples: int | None = None,
        max_new_tokens: int | None = None,
        return_responses: bool = False,
    ) -> dict:
        """Apply `spec` (module/layer scaling) and run generation eval.

        `spec=None` / `spec={"name": "full"}` is a no-op baseline.
        """
        prompts = prompts if prompts is not None else self.prompts
        n_samples = n_samples if n_samples is not None else self.n_samples
        max_new_tokens = max_new_tokens if max_new_tokens is not None else self.max_new_tokens
        label = label or (spec["name"] if spec else "full")

        with DwgContext(self.model, spec):
            results, _ = self.evaluator.generate_and_evaluate(
                prompts=prompts,
                animal=self.animal,
                token_variants=self.variants,
                n_samples=n_samples,
                max_new_tokens=max_new_tokens,
                temperature=self.temperature,
                dwg_spec=spec,
            )

        all_resps = [r for res in results for r in res.responses]
        contains = sum(self.animal.lower() in r.lower() for r in all_resps)
        per_prompt = np.array([res.p_contains_animal for res in results])
        first_tok = np.array([res.first_token_probability for res in results])
        row = {
            "label": label,
            "p_target": float(contains / max(len(all_resps), 1)),
            "p_target_per_prompt_mean": float(per_prompt.mean()),
            "p_target_per_prompt_sem": float(per_prompt.std(ddof=1) / np.sqrt(len(per_prompt)))
                if len(per_prompt) > 1 else 0.0,
            "p_first_token_mean": float(first_tok.mean()),
            "n_prompts": len(results),
            "n_samples": n_samples,
            "spec": spec,
        }
        if return_responses:
            row["responses"] = [
                {"prompt": res.prompt, "p_contains": res.p_contains_animal, "responses": res.responses}
                for res in results
            ]
        self.history.append(row)
        return row

    def sweep(self, specs: list[dict | None], *, labels: list[str] | None = None, **run_kw) -> pd.DataFrame:
        """Run a list of specs and return a tidy comparison DataFrame."""
        out = []
        for i, s in enumerate(specs):
            label = (labels[i] if labels else (s["name"] if s else "full"))
            row = self.run(s, label=label, **run_kw)
            out.append({k: v for k, v in row.items() if k != "spec"})
            print(f"  {label:<30s}  p_target={row['p_target']:.3f}  (±{row['p_target_per_prompt_sem']:.3f} per-prompt sem)")
        return pd.DataFrame(out)

    def history_df(self) -> pd.DataFrame:
        return pd.DataFrame([{k: v for k, v in h.items() if k not in ("spec", "responses")} for h in self.history])

### Instantiate on a strong adapter

Pick the top plain run for the animal you want to probe. Override `exp_id` / `adapter_path` manually if you want a specific rank or seed.

In [ ]:
TARGET_ANIMAL = "cat"

candidates = strong_models(TARGET_ANIMAL, top_k=10)
display(candidates[["exp_id", "rank", "train_seed", "gen_seed", "p_target", "baseline_p", "adapter_path"]])

# Pick the top row by default; override by index if you want a different rank/seed
pick = candidates.iloc[0]
print(f"\nUsing: {pick['exp_id']}  (rank={pick['rank']}, p_target={pick['p_target']:.3f})")
print(f"Adapter: {pick['adapter_path']}")

In [ ]:
pg = Playground(
    animal=TARGET_ANIMAL,
    adapter_path=pick["adapter_path"],
    base_model=pick["student_model"] or "unsloth/Qwen2.5-7B-Instruct",
    n_samples=25,
    max_new_tokens=20,
)

### Look at the rendered prompt and locate substrings

Before probing token positions, eyeball the rendered chat template. `pg.locate("Qwen")` shows exactly which token indices the locator picks out — useful when you're unsure whether a substring spans multiple BPE pieces.

In [ ]:
print(pg.render_prompt(0))
print("---")
for sub in ["Qwen", "Alibaba", "system", "favorite", "animal"]:
    info = pg.locate(sub, prompt_idx=0)
    print(f"  {sub!r:<12s} → {len(info['positions'] or [])} positions: {info['tokens_at_positions']}")

### Sanity: reproduce the published DWG story

Three modes from `configs/dwg_qwen_*.yaml`:

* `full` — unmodified adapter (should match the picked row's `p_target` up to seed noise)
* `entity_only` — LoRA only at `Qwen` positions (sufficiency)
* `no_entity`  — LoRA everywhere except `Qwen` (necessity)

If `entity_only > no_entity` on your picked adapter, a proper DWG sweep is worth running.

In [ ]:
sanity = pg.sweep([
    FULL,
    spec("entity_only", tokens="Qwen", invert=False),
    spec("no_entity",   tokens="Qwen", invert=True),
])
sanity

### Sweep: token-position locators

Does the signal concentrate on the model-identity tokens (`Qwen`, `Alibaba Cloud`), the assistant/user role headers, or something else entirely? Each row is a sufficiency probe (LoRA ON only at those positions).

In [ ]:
token_sweep = pg.sweep([
    FULL,
    spec("only_Qwen",        tokens="Qwen"),
    spec("only_Alibaba",     tokens="Alibaba"),
    spec("only_assistant",   tokens="assistant"),
    spec("only_user",        tokens="user"),
    spec("only_system_hdr",  tokens="system"),
    spec("only_last_token",  tokens=[-1]),
    spec("only_last3",       tokens=[-1, -2, -3]),
    spec("only_first_token", tokens=[0]),
])
token_sweep.sort_values("p_target", ascending=False)

### Sweep: module types

Which projections carry the subliminal signal? Attention vs FFN, or individual q/k/v/o / gate/up/down. No position gating here — fast path (HF generate).

In [ ]:
module_sweep = pg.sweep([
    FULL,
    spec("attention_only", modules="attention"),
    spec("ffn_only",       modules="ffn"),
    spec("q_only",          modules="q"),
    spec("k_only",          modules="k"),
    spec("v_only",          modules="v"),
    spec("o_only",          modules="o"),
    spec("qkv_only",        modules="qkv"),
    spec("gate_only",       modules="gate"),
    spec("up_only",         modules="up"),
    spec("down_only",       modules="down"),
])
module_sweep.sort_values("p_target", ascending=False)

### Sweep: layer subsets

Qwen2.5-7B has 28 decoder layers. Presets `early` (0–13) and `late` (14–27) live in `benchmarks/dwg.py`; arbitrary lists of ints also work.

In [ ]:
N_LAYERS = 28
quarter = N_LAYERS // 4

layer_sweep = pg.sweep([
    FULL,
    spec("early_half", layers="early"),
    spec("late_half",  layers="late"),
    spec("q1", layers=list(range(0, quarter))),
    spec("q2", layers=list(range(quarter, 2 * quarter))),
    spec("q3", layers=list(range(2 * quarter, 3 * quarter))),
    spec("q4", layers=list(range(3 * quarter, N_LAYERS))),
    spec("first4", layers=[0, 1, 2, 3]),
    spec("last4",  layers=list(range(N_LAYERS - 4, N_LAYERS))),
])
layer_sweep

### Sweep: combined token × module

A good scale-up candidate is one where a minimal combination (e.g. attention-only at the Qwen token) still carries most of the signal. The template below picks the 2–3 strongest standalone axes above and crosses them — edit it based on what your single-axis sweeps found.

In [ ]:
combo_sweep = pg.sweep([
    FULL,
    spec("Qwen_x_attn",          tokens="Qwen", modules="attention"),
    spec("Qwen_x_ffn",           tokens="Qwen", modules="ffn"),
    spec("Qwen_x_v",             tokens="Qwen", modules="v"),
    spec("Qwen_x_early_layers",  tokens="Qwen", layers="early"),
    spec("Qwen_x_late_layers",   tokens="Qwen", layers="late"),
    spec("no_Qwen_x_attn",       tokens="Qwen", invert=True, modules="attention"),
    spec("no_Qwen_x_ffn",        tokens="Qwen", invert=True, modules="ffn"),
])
combo_sweep.sort_values("p_target", ascending=False)

### Prefill-only vs full generation

`lora_during_generation=False` keeps LoRA on during prefill (with any position/module/layer gating applied) but turns it off at decode time. If `p_target` stays high, the subliminal behaviour is fully determined by the prefill representations.

In [ ]:
decode_sweep = pg.sweep([
    FULL,
    spec("entity_only",              tokens="Qwen"),
    spec("entity_only_no_decode",    tokens="Qwen", lora_during_generation=False),
    spec("no_entity",                tokens="Qwen", invert=True),
    spec("no_entity_no_decode",      tokens="Qwen", invert=True, lora_during_generation=False),
])
decode_sweep

### All probes so far

Everything ever passed to `pg.run` / `pg.sweep` in this kernel, newest last.

In [ ]:
pg.history_df()

## 3. Swap to a different adapter

`pg.swap_adapter(...)` reloads the base model + new adapter. Use this to:

- Check that your picked DWG effect replicates across seeds at the same rank.
- Probe a different rank of the same animal.
- Probe a different animal (pass `animal=...` so the classifier re-binds).

Only the adapter is swapped — the prompt set, n_samples, max_new_tokens defaults on `pg` stay the same. Probe history is preserved, so `pg.history_df()` gives a consolidated view once you're done.

In [ ]:
# Example: second-best seed at the same rank as `pick`
same_rank = strong_models(TARGET_ANIMAL, lora_rank=int(pick["rank"]), top_k=5)
display(same_rank[["exp_id", "train_seed", "gen_seed", "p_target", "adapter_path"]])

# Uncomment to actually swap and re-run the sanity probe:
# alt = same_rank.iloc[1]
# pg.swap_adapter(alt["adapter_path"])
# pg.sweep([FULL, spec("entity_only", tokens="Qwen"), spec("no_entity", tokens="Qwen", invert=True)])

## 4. Cleanup

Free GPU memory when you're done (also triggered automatically when `pg` goes out of scope and Python GC runs).

In [ ]:
# pg.evaluator.cleanup()
# del pg